# 🧠 Knowledge Agent v3 — 24/7 Omni-Source RAG & Full-Coverage Fine-Tuning Engine
> Zero-Link Sanitization + 100% Full-Coverage Chunking + Multi-Topic Telegram Dispatch

### 🚀 Telegram Delivery Topics
- 📄 **Raw PDFs** → [Topic 354](https://t.me/c/3958148223/354)
- 📝 **Clean Text** (Zero Links) → [Topic 355](https://t.me/c/3958148223/355)
- 📊 **Datasets** (Full Coverage JSONL) → [Topic 356](https://t.me/c/3958148223/356)
- 🚨 **General / Alerts / 24/7 Status** → [Topic 1](https://t.me/c/3958148223/1)


In [ ]:
# ── Step 1: Install Required Packages ─────────────────────────────
!pip install -q docling PyGithub requests thefuzz
print("✅ All dependencies installed successfully!")


In [ ]:
# ── Step 2: Configure Environment & Credentials ───────────────────
import os
def _get(k):
    try:
        from google.colab import userdata
        return userdata.get(k)
    except Exception:
        return None

CONFIG = {
    "GITHUB_TOKEN": _get("GITHUB_TOKEN") or "YOUR_GITHUB_TOKEN",
    "GITHUB_REPO_URL": _get("GITHUB_REPO_URL") or "https://github.com/Rawknowledge-database/knowledge",
    "TELEGRAM_BOT_TOKEN": _get("TELEGRAM_BOT_TOKEN") or "8525850416:AAGtYIM1sg8MF21_8lI2hOS1E-i9MosV4RE",
    "TELEGRAM_GROUP_ID": _get("TELEGRAM_GROUP_ID") or "-1003958148223",
    "RAW_PDF_TOPIC_ID": "354",
    "TEXT_MD_TOPIC_ID": "355",
    "DATASET_TOPIC_ID": "356",
    "GENERAL_TOPIC_ID": "1",
    "ADMIN_CHAT_ID": _get("ADMIN_CHAT_ID") or "6190001521",
    "HF_TOKEN": _get("HF_TOKEN") or "YOUR_HF_TOKEN",
    "PAPERS_PER_CATEGORY": "15"
}

for key, val in CONFIG.items():
    os.environ[key] = str(val)

print("✅ Credentials and Topics configured in Colab runtime!")


In [ ]:
# ── Step 3: Write Pipeline Engine (Self-Contained) ────────────────
%%writefile knowledge_agent.py
"""
🧠 Knowledge Agent v3 — 24/7 Omni-Source RAG & Full-Coverage Fine-Tuning Engine
================================================================================
Features:
• Zero-Link Sanitization : 100% clean markdown text files with zero URL pollution.
• 100% Full-Coverage Chunker: Splits documents into semantic 2K-token chunks (0% truncated on GPU).
• Multi-Topic Telegram Dispatch:
    - 📄 PDFs       -> Topic 354 (https://t.me/c/3958148223/354)
    - 📝 Text       -> Topic 355 (https://t.me/c/3958148223/355)
    - 📊 Datasets   -> Topic 356 (https://t.me/c/3958148223/356)
    - 🚨 General    -> Topic 1   (https://t.me/c/3958148223/1)
• GitHub Dual-Vault Storage:
    - text_vault/       <- Clean full-text markdown (Zero Links)
    - dataset_vault/    <- Full-coverage chunked JSONL for LLM training
    - README.md         <- Master catalog & documentation
• 4-Layer Deduplication : L1 (arXiv ID) + L2 (Title) + L3 (Fuzzy) + L4 (SHA-256)
• 24/7 Crash-Proof      : Continuous daemon loop with automatic alert dispatch to Topic 1
"""

from __future__ import annotations

import hashlib
import html
import json
import logging
import os
import re
import subprocess
import sys
import time
import xml.etree.ElementTree as ET
from dataclasses import dataclass, field
from datetime import datetime, timezone
from pathlib import Path

# Ensure UTF-8 output on Windows terminal
if sys.platform == "win32":
    try:
        sys.stdout.reconfigure(encoding="utf-8")
        sys.stderr.reconfigure(encoding="utf-8")
    except Exception:
        pass

# ── Auto-install missing packages in Google Colab / Fresh environments ─────────
for _mod, _pkg in [
    ("docling", "docling"),
    ("github", "PyGithub"),
    ("thefuzz", "thefuzz"),
    ("requests", "requests"),
]:
    try:
        __import__(_mod)
    except ImportError:
        print(f"📦 Auto-installing missing package '{_pkg}'...", flush=True)
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", _pkg])

import requests
from docling.datamodel.pipeline_options import PdfPipelineOptions
from docling.document_converter import DocumentConverter, PdfFormatOption
from github import Github, GithubException, Auth
from thefuzz import fuzz

# ── Paths & Colab / Jupyter Safe Root ──────────────────────────────────────────
try:
    ROOT = Path(__file__).parent
except NameError:
    ROOT = Path.cwd()

TEXT_VAULT_DIR = ROOT / "text_vault"
DATASET_VAULT_DIR = ROOT / "dataset_vault"
PDF_DIR = ROOT / "raw_pdfs"
REGISTRY = ROOT / "registry.json"
LOG_DIR = ROOT / "logs"
GLOBAL_DATASET_FILE = DATASET_VAULT_DIR / "dataset.jsonl"

TEXT_VAULT_DIR.mkdir(exist_ok=True)
DATASET_VAULT_DIR.mkdir(exist_ok=True)
PDF_DIR.mkdir(exist_ok=True)
LOG_DIR.mkdir(exist_ok=True)

# ── Logging Setup ──────────────────────────────────────────────────────────────
log_file = LOG_DIR / f"run_{datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S')}.log"
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    handlers=[logging.FileHandler(log_file, encoding="utf-8"), logging.StreamHandler(sys.stdout)],
    force=True,
)
log = logging.getLogger(__name__)

for noisy in ["MatchingPostProcessor", "docling", "docling.pipeline", "urllib3", "github"]:
    logging.getLogger(noisy).setLevel(logging.ERROR)


def status(msg: str, *args) -> None:
    """Print directly to terminal/Colab with flush=True so logs are never hidden."""
    if args:
        try:
            formatted = msg % args
        except Exception:
            formatted = f"{msg} " + " ".join(str(a) for a in args)
    else:
        formatted = msg
    print(formatted, flush=True)
    log.info(formatted)


# ══════════════════════════════════════════════════════════════════════════════
#  DATA MODEL
# ══════════════════════════════════════════════════════════════════════════════

@dataclass
class PaperInfo:
    """Unified metadata container. Works with RSS, HF API, and direct lookups."""
    arxiv_id: str
    title: str
    abstract: str = ""
    authors: str = ""
    categories: str = ""
    published: str = ""
    pdf_url: str = ""
    entry_url: str = ""
    code_repo: str | None = None
    stars: int | None = None
    upvotes: int | None = None
    track: str = ""

    def __post_init__(self):
        base = strip_arxiv_version(self.arxiv_id)
        if not self.pdf_url:
            self.pdf_url = f"https://arxiv.org/pdf/{base}.pdf"
        if not self.entry_url:
            self.entry_url = f"https://arxiv.org/abs/{base}"


# ══════════════════════════════════════════════════════════════════════════════
#  TRACK CURRICULUM
# ══════════════════════════════════════════════════════════════════════════════

TECH_CURRICULUM_RSS = [
    ("🛡 Cybersecurity & Exploits",           ["cs.CR"]),
    ("🤖 AI & Language Models (LLMs)",        ["cs.AI", "cs.CL"]),
    ("🧠 Machine Learning & Deep Learning",   ["cs.LG", "stat.ML"]),
    ("👁 Computer Vision & Multimodal",        ["cs.CV"]),
    ("💻 Advanced Coding & Algorithms",       ["cs.DS", "cs.PL"]),
    ("🛠 Software Eng & Git Architecture",     ["cs.SE"]),
    ("🐧 Linux & Operating Systems",           ["cs.OS"]),
    ("☁ Cloud, Servers & Distributed Sys",    ["cs.DC"]),
    ("🌐 Networks, Web & Protocols",          ["cs.NI", "cs.IR"]),
    ("📐 Computational Mathematics",           ["math.NA", "math.OC", "cs.CC"]),
]

PAPERS_PER_CATEGORY = int(os.environ.get("PAPERS_PER_CATEGORY", "15"))
FUZZY_THRESHOLD = 88
MIN_MD_CHARS = 400

TELEGRAM_API = "https://api.telegram.org/bot{token}/{method}"


# ══════════════════════════════════════════════════════════════════════════════
#  REGISTRY  —  Load / Save / Remote Sync
# ══════════════════════════════════════════════════════════════════════════════

def load_registry() -> dict:
    if REGISTRY.exists():
        try:
            return json.loads(REGISTRY.read_text(encoding="utf-8"))
        except Exception:
            return {"papers": [], "hashes": [], "titles": []}
    return {"papers": [], "hashes": [], "titles": []}


def save_registry(reg: dict) -> None:
    REGISTRY.write_text(json.dumps(reg, indent=2), encoding="utf-8")


def sync_remote_registry(gh_token: str, repo_url: str, local_reg: dict) -> dict:
    """Pull the latest remote registry from GitHub so documents are NEVER re-downloaded even across machines."""
    try:
        repo_name = re.sub(r"https?://github\.com/", "", repo_url).rstrip("/").removesuffix(".git")
        g = Github(auth=Auth.Token(gh_token))
        repo = g.get_repo(repo_name)
        remote_file = repo.get_contents("registry.json")
        remote_reg = json.loads(remote_file.decoded_content.decode("utf-8"))
        merged = {
            "papers": list(dict.fromkeys(local_reg.get("papers", []) + remote_reg.get("papers", []))),
            "hashes": list(dict.fromkeys(local_reg.get("hashes", []) + remote_reg.get("hashes", []))),
            "titles": list(dict.fromkeys(local_reg.get("titles", []) + remote_reg.get("titles", []))),
        }
        save_registry(merged)
        status("🔄 Registry synced with GitHub vault: %d existing papers known", len(merged["papers"]))
        return merged
    except Exception as e:
        status("⚠️ Could not sync remote registry from GitHub: %s. Using local.", e)
        return local_reg


# ══════════════════════════════════════════════════════════════════════════════
#  4-LAYER DEDUPLICATION GUARD
# ══════════════════════════════════════════════════════════════════════════════

def clean_str(s: str) -> str:
    return re.sub(r"[^a-zA-Z0-9 ]", "", s.lower()).strip()


def strip_arxiv_version(aid: str) -> str:
    """Normalize '2609.04199v1' -> '2609.04199' to prevent version duplicates."""
    return re.sub(r"v\d+$", "", aid.strip())


base_arxiv_id = strip_arxiv_version


def is_duplicate(arxiv_id: str, title: str, pdf_hash: str | None, reg: dict) -> str | None:
    """Bulletproof duplicate check:
    Layer 1: Exact or version-stripped arXiv ID
    Layer 2: Exact normalized title match
    Layer 3: High-similarity fuzzy title match
    Layer 4: SHA-256 PDF hash match
    """
    known_papers = reg.get("papers", [])
    known_titles = reg.get("titles", [])
    known_hashes = reg.get("hashes", [])

    # Layer 1 — arXiv ID & Base ID
    curr_base = base_arxiv_id(arxiv_id)
    for p in known_papers:
        if p == arxiv_id or base_arxiv_id(p) == curr_base:
            return f"Layer-1: ID '{arxiv_id}' matches existing '{p}'"

    # Layer 2 — Exact normalized title match
    clean_title = clean_str(title)
    clean_known = [clean_str(t) for t in known_titles]
    if clean_title in clean_known:
        idx = clean_known.index(clean_title)
        return f"Layer-2: Title matches exactly with '{known_titles[idx]}'"

    # Layer 3 — Fuzzy title match
    for existing_title in known_titles:
        score = fuzz.ratio(clean_title, clean_str(existing_title))
        if score >= FUZZY_THRESHOLD:
            return f"Layer-3: Title fuzzy match {score}% vs '{existing_title}'"

    # Layer 4 — PDF content SHA-256 hash
    if pdf_hash and pdf_hash in known_hashes:
        return f"Layer-4: Exact PDF SHA-256 ({pdf_hash[:12]}…) already in registry"

    return None


# ══════════════════════════════════════════════════════════════════════════════
#  ZERO-LINK SANITIZATION ENGINE (Pristine Text Guarantee)
# ══════════════════════════════════════════════════════════════════════════════

def strip_all_links(text: str) -> str:
    """Strip all web links (markdown links, image embeds, HTML hrefs, and raw URLs)
    from text to ensure zero link pollution for RAG, text files, and LLM fine-tuning.
    """
    if not text:
        return ""

    # 1. Strip markdown image embeds: ![alt](url) -> ''
    text = re.sub(r'!\[[^\]]*\]\([^\)]+\)', '', text)

    # 2. Strip GitHub badge syntax like [![badge](url)](link)
    text = re.sub(r'\[!\[[^\]]*\]\([^\)]+\)\]\([^\)]+\)', '', text)

    # 3. Strip markdown links: [anchor](url) -> anchor
    text = re.sub(r'\[([^\]]+)\]\([^\)]+\)', r'\1', text)

    # 4. Strip HTML anchor links: <a href="...">anchor</a> -> anchor
    text = re.sub(r'<a\s+[^>]*href=[^>]*>(.*?)</a>', r'\1', text, flags=re.DOTALL | re.IGNORECASE)

    # 5. Strip common HTML wrapper tags but keep text
    text = re.sub(r'</?(?:details|summary|div|span|p)[^>]*>', '', text, flags=re.IGNORECASE)

    # 6. Strip enclosed URLs: <http://...> or <https://...> -> ''
    text = re.sub(r'<https?://[^>]+>', '', text)

    # 7. Strip raw URLs: http://, https://, ftp://
    text = re.sub(r'https?://[^\s\)\]]+', '', text)
    text = re.sub(r'ftp://[^\s\)\]]+', '', text)

    # 8. Strip web URLs starting with www.
    text = re.sub(r'www\.[^\s\)\]]+', '', text)

    # 9. Clean up empty brackets or parentheses left behind: () or []
    text = re.sub(r'\(\s*\)', '', text)
    text = re.sub(r'\[\s*\]', '', text)

    # 10. Collapse multiple spaces and excessive empty lines
    text = re.sub(r'[ \t]{2,}', ' ', text)
    text = re.sub(r'\n{3,}', '\n\n', text)

    return text.strip()


# ══════════════════════════════════════════════════════════════════════════════
#  SOURCES:  arXiv RSS + Hugging Face Trending
# ══════════════════════════════════════════════════════════════════════════════

def parse_arxiv_rss(categories: list[str], limit: int = 30) -> list[PaperInfo]:
    """Fetch papers from arXiv RSS feeds — completely bypasses export.arxiv.org API.
    These feeds are served from a CDN and have zero rate-limiting.
    """
    papers: list[PaperInfo] = []
    seen_ids: set[str] = set()

    for cat in categories:
        rss_url = f"https://rss.arxiv.org/rss/{cat}"
        try:
            resp = requests.get(rss_url, headers={"User-Agent": "KnowledgeBot/3.0"}, timeout=20)
            resp.raise_for_status()
            root = ET.fromstring(resp.content)

            dc_ns = "{http://purl.org/dc/elements/1.1/}"

            for item in root.findall(".//item"):
                raw_title = (item.findtext("title") or "").strip()
                link = (item.findtext("link") or "").strip()
                description = (item.findtext("description") or "").strip()

                arxiv_id_match = re.search(r"arxiv\.org/abs/([\d.]+)", link)
                if not arxiv_id_match:
                    continue
                arxiv_id = arxiv_id_match.group(1)
                base_id = strip_arxiv_version(arxiv_id)

                if base_id in seen_ids:
                    continue
                seen_ids.add(base_id)

                clean_title = re.sub(r"\s*\(arXiv:[\d.]+v?\d*\s*\[.*?\]\)\s*$", "", raw_title).strip()
                if not clean_title:
                    clean_title = raw_title

                authors = (item.findtext(f"{dc_ns}creator") or "").strip()

                papers.append(PaperInfo(
                    arxiv_id=base_id,
                    title=clean_title,
                    abstract=description[:500],
                    authors=authors,
                    categories=cat,
                    published=datetime.now(timezone.utc).strftime("%Y-%m-%d"),
                ))

            status("  📡 RSS %s: fetched %d items", cat, len([p for p in papers if p.categories == cat]))
            time.sleep(0.4)

        except Exception as e:
            status("  ⚠️ RSS feed error for %s: %s", cat, e)

    return papers[:limit]


def fetch_huggingface_trending(hf_token: str, limit: int = 20) -> list[PaperInfo]:
    """Fetch top trending AI research papers from Hugging Face Daily Papers."""
    papers: list[PaperInfo] = []
    headers = {"User-Agent": "KnowledgeBot/3.0"}
    if hf_token:
        headers["Authorization"] = f"Bearer {hf_token}"

    try:
        r = requests.get("https://huggingface.co/api/daily_papers", headers=headers, timeout=20)
        if r.status_code == 200:
            for item in r.json()[:limit]:
                paper_info = item.get("paper", {})
                raw_id = paper_info.get("id")
                if not raw_id:
                    continue

                base_id = strip_arxiv_version(raw_id)
                title = paper_info.get("title", "Unknown")
                abstract = paper_info.get("summary", "")[:500]
                authors_list = paper_info.get("authors", [])
                if isinstance(authors_list, list):
                    authors = ", ".join(
                        a.get("name", a) if isinstance(a, dict) else str(a)
                        for a in authors_list[:5]
                    )
                else:
                    authors = str(authors_list)

                code_repo = paper_info.get("githubRepo")
                stars = paper_info.get("githubStars")
                upvotes = item.get("numUpvotes") or paper_info.get("upvotes")

                papers.append(PaperInfo(
                    arxiv_id=base_id,
                    title=title,
                    abstract=abstract,
                    authors=authors,
                    published=paper_info.get("publishedAt", "")[:10],
                    code_repo=code_repo,
                    stars=stars,
                    upvotes=upvotes,
                    track="🔥 Hugging Face Trending",
                ))

            status("  🔥 HF Trending: fetched %d papers (authenticated=%s)", len(papers), bool(hf_token))
        else:
            status("  ⚠️ HF API returned HTTP %d", r.status_code)
    except Exception as e:
        log.warning("Could not fetch Hugging Face trending papers: %s", e)

    return papers


# ══════════════════════════════════════════════════════════════════════════════
#  PDF DOWNLOAD & DOCLING ENGINE
# ══════════════════════════════════════════════════════════════════════════════

def download_pdf(paper: PaperInfo, dest_dir: Path, max_retries: int = 3) -> tuple[Path, str]:
    """Stream download PDF and compute SHA-256 hash on the fly with retry logic."""
    safe_id = paper.arxiv_id.replace("/", "_")
    pdf_path = dest_dir / f"{safe_id}.pdf"

    for attempt in range(max_retries):
        hasher = hashlib.sha256()
        try:
            with requests.get(
                paper.pdf_url,
                headers={"User-Agent": "KnowledgeBot/3.0"},
                stream=True,
                timeout=45,
            ) as resp:
                resp.raise_for_status()
                with open(pdf_path, "wb") as f:
                    for chunk in resp.iter_content(chunk_size=65536):
                        if chunk:
                            f.write(chunk)
                            hasher.update(chunk)
            sha256 = hasher.hexdigest()
            return pdf_path, sha256
        except Exception as exc:
            if pdf_path.exists():
                pdf_path.unlink(missing_ok=True)
            if attempt < max_retries - 1:
                wait_sec = 2.0 * (attempt + 1)
                log.warning("PDF download retry %d/%d for %s (%.1fs): %s", attempt + 1, max_retries, safe_id, wait_sec, exc)
                time.sleep(wait_sec)
            else:
                raise exc


_converter: DocumentConverter | None = None


def get_converter() -> DocumentConverter:
    """Initialize fast zero-OCR Docling converter once and reuse in memory."""
    global _converter
    if _converter is None:
        log.info("Initializing fast Docling pipeline (zero-OCR digital mode)...")
        pipeline_options = PdfPipelineOptions(do_ocr=False, do_table_structure=True)
        _converter = DocumentConverter(
            format_options={"pdf": PdfFormatOption(pipeline_options=pipeline_options)}
        )
    return _converter


def convert_to_markdown(pdf_path: Path) -> tuple[str, int, int]:
    """Convert PDF to structured Markdown using fast cached Docling engine."""
    converter = get_converter()
    result = converter.convert(str(pdf_path))
    md_text = result.document.export_to_markdown()

    word_count = len(md_text.split())
    try:
        page_count = len(result.document.pages)
    except Exception:
        page_count = max(1, word_count // 300)

    return md_text, word_count, page_count


def strip_bibliography(md_text: str) -> tuple[str, str]:
    """Detect and separate the bibliography/references section from the main content."""
    bib_patterns = [
        r"^#{1,3}\s*References?\s*$",
        r"^#{1,3}\s*Bibliography\s*$",
        r"^#{1,3}\s*Works?\s+Cited\s*$",
        r"^#{1,3}\s*Literature\s*$",
        r"^\*\*References?\*\*\s*$",
    ]
    combined_pattern = "|".join(bib_patterns)
    lines = md_text.split("\n")

    bib_start = None
    for i in range(len(lines) - 1, max(len(lines) // 2, 0), -1):
        if re.match(combined_pattern, lines[i].strip(), re.IGNORECASE):
            bib_start = i
            break

    if bib_start is not None:
        clean_content = "\n".join(lines[:bib_start]).rstrip()
        bibliography = "\n".join(lines[bib_start:])
        return clean_content, bibliography

    return md_text, ""


def extract_code_repo(text: str, abstract: str = "") -> str | None:
    """Extract first valid companion GitHub repository link found in markdown or abstract."""
    full_text = f"{abstract}\n{text[:4000]}"
    matches = re.findall(r"github\.com/([A-Za-z0-9_.-]+/[A-Za-z0-9_.-]+)", full_text, re.IGNORECASE)
    ignored = {"docling", "pytorch", "huggingface", "tensorflow", "keras", "numpy", "scipy", "docling-project"}
    for m in matches:
        clean_m = m.rstrip("/").removesuffix(".git")
        parts = clean_m.split("/")
        if len(parts) == 2 and parts[0].lower() not in ignored and parts[1].lower() not in ignored:
            return f"https://github.com/{clean_m}"
    return None


def fetch_companion_code(code_repo_url: str, gh_token: str) -> str | None:
    """Fetch the README.md from a paper's companion GitHub repository."""
    if not code_repo_url or not gh_token:
        return None
    try:
        match = re.match(r"https?://github\.com/([A-Za-z0-9_.-]+/[A-Za-z0-9_.-]+)", code_repo_url)
        if not match:
            return None
        repo_name = match.group(1).rstrip("/").removesuffix(".git")
        g = Github(auth=Auth.Token(gh_token))
        repo = g.get_repo(repo_name)

        readme_content = None
        for readme_name in ["README.md", "readme.md", "README.rst", "README"]:
            try:
                readme_file = repo.get_contents(readme_name)
                readme_content = readme_file.decoded_content.decode("utf-8")
                break
            except GithubException:
                continue

        if readme_content:
            return f"\n\n## Companion Repository Code (GitHub)\n\n{readme_content}"
        return None
    except Exception as e:
        log.warning("Could not fetch companion code for %s: %s", code_repo_url, e)
        return None


# ══════════════════════════════════════════════════════════════════════════════
#  ZERO-LINK FRONTMATTER & FULL-COVERAGE CHUNKER
# ══════════════════════════════════════════════════════════════════════════════

def build_zero_link_frontmatter(
    paper: PaperInfo,
    kb_id: str,
    word_count: int,
    page_count: int,
    total_chunks: int,
) -> str:
    """Generate pure YAML frontmatter with ZERO URLs, ensuring 100% clean RAG metadata."""
    safe_title = paper.title.replace('"', "'")
    safe_authors = paper.authors[:200].replace('"', "'")
    now_iso = datetime.now(timezone.utc).strftime('%Y-%m-%dT%H:%M:%SZ')

    return f"""---
kb_id: "{kb_id}"
arxiv_id: "{paper.arxiv_id}"
title: "{safe_title}"
track: "{paper.track}"
categories: "{paper.categories}"
authors: "{safe_authors}"
published: "{paper.published}"
ingested: "{now_iso}"
word_count: {word_count}
page_count: {page_count}
total_chunks: {total_chunks}
has_code: {str(bool(paper.code_repo)).lower()}
---

"""


def chunk_document_full_coverage(
    clean_text: str,
    kb_id: str,
    paper: PaperInfo,
    target_words_per_chunk: int = 1500,
    overlap_words: int = 120,
) -> list[dict]:
    """100% Full-Coverage Smart Chunker:
    Splits the document into natural semantic chunks (~2000 tokens) so that:
    1. Zero text is truncated or lost during GPU fine-tuning.
    2. GPU training does not run out of memory (OOM).
    3. The model absorbs 100% of the paper's contents.
    """
    paragraphs = re.split(r"\n\s*\n", clean_text)
    chunks_text: list[str] = []
    curr_chunk: list[str] = []
    curr_words = 0

    for para in paragraphs:
        p = para.strip()
        if not p:
            continue
        p_words = len(p.split())

        if curr_words + p_words > target_words_per_chunk and curr_chunk:
            combined = "\n\n".join(curr_chunk)
            chunks_text.append(combined)

            # Extract overlap words from end of current chunk
            all_words = combined.split()
            overlap_text = " ".join(all_words[-overlap_words:]) if len(all_words) > overlap_words else ""
            curr_chunk = [overlap_text, p] if overlap_text else [p]
            curr_words = len(overlap_text.split()) + p_words
        else:
            curr_chunk.append(p)
            curr_words += p_words

    if curr_chunk:
        chunks_text.append("\n\n".join(curr_chunk))

    total_chunks = len(chunks_text)
    records: list[dict] = []

    for idx, c_text in enumerate(chunks_text, start=1):
        chunk_id = f"{kb_id}_c{idx:02d}"
        records.append({
            "id": chunk_id,
            "kb_id": kb_id,
            "chunk_index": idx,
            "total_chunks": total_chunks,
            "title": paper.title,
            "track": paper.track,
            "categories": paper.categories,
            "authors": paper.authors,
            "published": paper.published,
            "text": c_text.strip(),
        })

    return records


# ══════════════════════════════════════════════════════════════════════════════
#  TELEGRAM DISPATCHER (Topic 354: PDF, Topic 355: Text, Topic 356: Dataset, Topic 1: General)
# ══════════════════════════════════════════════════════════════════════════════

def tg_send(token: str, chat_id: str, text: str, **kwargs) -> requests.Response:
    url = TELEGRAM_API.format(token=token, method="sendMessage")
    tid = kwargs.pop("message_thread_id", None)
    payload = {"chat_id": chat_id, "text": text, "parse_mode": "HTML", **kwargs}
    if tid is not None:
        try:
            tid_int = int(tid)
            if tid_int > 1:
                payload["message_thread_id"] = tid_int
        except (ValueError, TypeError):
            pass
    r = requests.post(url, json=payload, timeout=30)
    r.raise_for_status()
    return r


def tg_send_doc(
    token: str,
    chat_id: str,
    file_path: Path,
    caption: str,
    reply_markup: dict | None = None,
    **kwargs,
) -> requests.Response:
    url = TELEGRAM_API.format(token=token, method="sendDocument")
    tid = kwargs.pop("message_thread_id", None)
    data = {"chat_id": chat_id, "caption": caption[:1024], "parse_mode": "HTML", **kwargs}
    if tid is not None:
        try:
            tid_int = int(tid)
            if tid_int > 1:
                data["message_thread_id"] = tid_int
        except (ValueError, TypeError):
            pass
    if reply_markup:
        data["reply_markup"] = json.dumps(reply_markup)

    with open(file_path, "rb") as f:
        files = {"document": (file_path.name, f, "application/octet-stream")}
        r = requests.post(url, data=data, files=files, timeout=120)

    r.raise_for_status()
    return r


def build_pdf_card(paper: PaperInfo, kb_id: str) -> str:
    safe_title = html.escape(paper.title.strip().replace("\n", " "))
    authors = html.escape(paper.authors[:100])
    abstract = html.escape(paper.abstract[:280] + "…" if len(paper.abstract) > 280 else paper.abstract)

    card = (
        f"📄 <b>[PDF: {kb_id}]</b>\n"
        f"<b>{safe_title}</b>\n\n"
        f"🏷 <b>Track:</b> {paper.track}\n"
        f"👥 <i>{authors}</i> ({paper.published or 'Recent'})\n"
        f"🗂 <code>{paper.categories}</code>\n\n"
        f"📝 <b>Abstract:</b>\n{abstract}"
    )
    return card


def build_text_card(paper: PaperInfo, kb_id: str, word_count: int, page_count: int) -> str:
    safe_title = html.escape(paper.title.strip().replace("\n", " "))
    card = (
        f"📝 <b>[TEXT: {kb_id}]</b> (Clean Markdown)\n"
        f"<b>{safe_title}</b>\n\n"
        f"🏷 <b>Track:</b> {paper.track}\n"
        f"📊 <b>Size:</b> {word_count:,} words | {page_count} pages\n"
        f"✨ <b>Zero-Link Sanitized:</b> 100% pure text without URL clutter\n"
        f"🎯 <b>RAG-Ready:</b> Metadata frontmatter included"
    )
    return card


def build_dataset_card(paper: PaperInfo, kb_id: str, total_chunks: int, total_words: int) -> str:
    safe_title = html.escape(paper.title.strip().replace("\n", " "))
    card = (
        f"📊 <b>[DATASET: {kb_id}]</b> (Fine-Tuning JSONL)\n"
        f"<b>{safe_title}</b>\n\n"
        f"🏷 <b>Track:</b> {paper.track}\n"
        f"🧩 <b>Full Coverage:</b> {total_chunks} Chunks ({total_words:,} words total)\n"
        f"⚡ <b>Zero Truncation:</b> Sized for 2K-4K token GPU context\n"
        f"🤖 <b>Training Ready:</b> Compatible with Unsloth, Axolotl, LLaMA-Factory"
    )
    return card


def send_general_alert(token: str, chat_id: str, text: str, general_topic: int = 1) -> None:
    """Send health and error notifications to the General Topic (#1)."""
    try:
        tg_send(token, chat_id, text, message_thread_id=general_topic)
    except Exception as e:
        log.warning("Could not send to General Topic (%d): %s", general_topic, e)


# ══════════════════════════════════════════════════════════════════════════════
#  GITHUB DUAL-VAULT & MASTER README SYNC
# ══════════════════════════════════════════════════════════════════════════════

def push_or_update(repo, file_path: str, commit_msg: str, content: str) -> None:
    """Safely create or update file with 404 check and sha-conflict retry."""
    for attempt in range(3):
        try:
            try:
                existing = repo.get_contents(file_path)
                repo.update_file(file_path, commit_msg, content, existing.sha)
                return
            except GithubException as ge:
                if ge.status == 404:
                    repo.create_file(file_path, commit_msg, content)
                    return
                time.sleep(1.5)
        except Exception:
            if attempt == 2:
                raise
            time.sleep(1.5)


def generate_master_readme(reg: dict, github_repo_url: str) -> str:
    """Generate comprehensive Master README for the GitHub Vault."""
    total_papers = len(reg.get("papers", []))
    now_str = datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M UTC")

    catalog_lines = []
    titles = reg.get("titles", [])
    papers = reg.get("papers", [])

    for i in range(max(0, len(papers) - 30), len(papers)):
        pid = f"KB-{i+1:04d}"
        t = titles[i] if i < len(titles) else "Research Paper"
        aid = papers[i]
        catalog_lines.append(f"| `{pid}` | [{t}](text_vault/{pid}.md) | `{aid}` | [JSONL Dataset](dataset_vault/{pid}_dataset.jsonl) |")

    catalog_table = "\n".join(catalog_lines) if catalog_lines else "| - | No papers registered yet | - | - |"
    tb = chr(96) * 3

    return f"""# 🧠 Knowledge Vault — Master Research & Fine-Tuning Corpus

An automated, 24/7 autonomous research ingestion vault. Contains zero-link sanitized documents and 100% full-coverage chunked datasets for LLM fine-tuning.

### 📊 Vault Overview
- **Total Ingested Documents**: `{total_papers}`
- **Last Sync**: `{now_str}`
- **Repository**: `{github_repo_url}`

---

## 📁 Repository Structure

{tb}
├── text_vault/              # 📝 100% Zero-Link Clean Markdown Files (RAG-Ready)
│   ├── KB-0001.md
│   └── ...
├── dataset_vault/           # 📊 Full-Coverage Chunked JSONL (Fine-Tuning Ready)
│   ├── KB-0001_dataset.jsonl
│   └── dataset.jsonl        # Global master training dataset
├── registry.json            # 🛡️ 4-Layer Deduplication Database
└── README.md                # 📖 Master Catalog & Index
{tb}

---

## 🚀 Telegram Real-Time Delivery Topics

| Content Type | Telegram Topic | Description |
| :--- | :--- | :--- |
| **📄 Raw PDFs** | [Topic 354](https://t.me/c/3958148223/354) | Original research PDFs |
| **📝 Clean Text** | [Topic 355](https://t.me/c/3958148223/355) | Zero-link sanitized markdown for RAG |
| **📊 Datasets** | [Topic 356](https://t.me/c/3958148223/356) | Chunked fine-tuning JSONL (Unsloth ready) |
| **🚨 General / Alerts** | [Topic 1](https://t.me/c/3958148223/1) | System health, 24/7 cycle status, error alerts |

---

## 📚 Recent Ingested Catalog

| ID | Title | arXiv ID | Dataset |
| :--- | :--- | :--- | :--- |
{catalog_table}
"""


def github_save_vault(
    gh_token: str,
    repo_url: str,
    kb_id: str,
    clean_md_text: str,
    dataset_jsonl_text: str,
    reg: dict,
) -> None:
    """Push text file, dataset file, registry, and master README to GitHub Vault."""
    repo_name = re.sub(r"https?://github\.com/", "", repo_url).rstrip("/").removesuffix(".git")
    auth = Auth.Token(gh_token)
    g = Github(auth=auth)
    repo = g.get_repo(repo_name)

    text_path = f"text_vault/{kb_id}.md"
    dataset_path = f"dataset_vault/{kb_id}_dataset.jsonl"
    reg_path = "registry.json"
    readme_path = "README.md"
    commit_msg = f"[{kb_id}] Ingest text & dataset into Knowledge Vault"

    # Push files
    push_or_update(repo, text_path, commit_msg, clean_md_text)
    push_or_update(repo, dataset_path, commit_msg, dataset_jsonl_text)
    push_or_update(repo, reg_path, commit_msg, json.dumps(reg, indent=2))

    # Update Master README
    master_readme = generate_master_readme(reg, repo_url)
    push_or_update(repo, readme_path, f"Update Master Catalog for {kb_id}", master_readme)

    log.info("GitHub: Pushed %s, %s, and updated Master README.md", text_path, dataset_path)


# ══════════════════════════════════════════════════════════════════════════════
#  ID GENERATOR
# ══════════════════════════════════════════════════════════════════════════════

def next_kb_id(reg: dict) -> str:
    n = len(reg.get("papers", [])) + 1
    return f"KB-{n:04d}"


# ══════════════════════════════════════════════════════════════════════════════
#  MAIN PIPELINE RUNNER
# ══════════════════════════════════════════════════════════════════════════════

def run(env: dict) -> dict:
    token = env["TELEGRAM_BOT_TOKEN"]
    group_id = env["TELEGRAM_GROUP_ID"]
    raw_topic = int(env.get("RAW_PDF_TOPIC_ID", 354))
    md_topic = int(env.get("TEXT_MD_TOPIC_ID", 355))
    dataset_topic = int(env.get("DATASET_TOPIC_ID", 356))
    general_topic = int(env.get("GENERAL_TOPIC_ID", 1))

    admin_id = env["ADMIN_CHAT_ID"]
    gh_token = env["GITHUB_TOKEN"]
    gh_repo = env["GITHUB_REPO_URL"]
    hf_token = env.get("HF_TOKEN", "")

    # 1. Sync remote registry from GitHub
    reg = load_registry()
    reg = sync_remote_registry(gh_token, gh_repo, reg)

    stats = {"scanned": 0, "skipped": 0, "added": 0, "errors": 0}
    start_time = time.time()
    papers_per_cat = int(env.get("PAPERS_PER_CATEGORY", PAPERS_PER_CATEGORY))

    status("════════════════════════════════════════════════════════════════════════════")
    status("🧠 Knowledge Agent v3 — 24/7 Omni-Source & Full-Coverage Fine-Tuning Engine")
    status("   • Target Topics : PDF (#%d) | Text (#%d) | Dataset (#%d) | General (#%d)",
           raw_topic, md_topic, dataset_topic, general_topic)
    status("   • Batch Size    : %d/track | Known Registry: %d papers", papers_per_cat, len(reg.get("papers", [])))
    status("════════════════════════════════════════════════════════════════════════════")

    # 2. Build master candidates list from all sources
    all_batches: list[tuple[str, list[PaperInfo]]] = []

    # Source 1: Hugging Face Trending
    status("\n🔥 [Discovery] Fetching Hugging Face Daily Trending Papers...")
    hf_papers = fetch_huggingface_trending(hf_token, limit=15)
    if hf_papers:
        all_batches.append(("🔥 Hugging Face Trending", hf_papers))

    # Source 2: arXiv RSS Feeds
    for track_name, categories in TECH_CURRICULUM_RSS:
        status("\n📡 [Discovery] Fetching RSS feed for: %s", track_name)
        rss_papers = parse_arxiv_rss(categories, limit=papers_per_cat)
        for p in rss_papers:
            p.track = track_name
        if rss_papers:
            all_batches.append((track_name, rss_papers))

    # 3. Process each batch
    for batch_name, papers in all_batches:
        status("\n📚 [Track Processing] %s (%d candidates)", batch_name, len(papers))

        for paper in papers:
            stats["scanned"] += 1
            status("\n  ──────────────────────────────────────────────────────────")
            status("  🔍 [%s] %s", paper.arxiv_id, paper.title[:70])

            # Pre-download Deduplication (Layers 1, 2, 3)
            dup_reason = is_duplicate(paper.arxiv_id, paper.title, None, reg)
            if dup_reason:
                status("    ⏭ SKIP (Pre-download): %s", dup_reason)
                stats["skipped"] += 1
                continue

            pdf_path: Path | None = None
            pdf_hash: str | None = None

            try:
                # Step 1: Download PDF
                t_dl = time.time()
                pdf_path, pdf_hash = download_pdf(paper, PDF_DIR)
                dl_duration = max(0.1, time.time() - t_dl)
                pdf_size_mb = pdf_path.stat().st_size / (1024 * 1024)
                status("    ⬇ Downloaded: %.2f MB in %.1fs (%.2f MB/s)",
                       pdf_size_mb, dl_duration, pdf_size_mb / dl_duration)

                # Step 2: Post-download Deduplication (Layer 4 — SHA-256)
                dup_reason = is_duplicate(paper.arxiv_id, paper.title, pdf_hash, reg)
                if dup_reason:
                    status("    ⏭ SKIP (Hash duplicate): %s", dup_reason)
                    stats["skipped"] += 1
                    pdf_path.unlink(missing_ok=True)
                    continue

                # Step 3: Docling PDF Conversion
                t_conv = time.time()
                status("    ⚡ Converting with Docling GPU/digital engine...")
                raw_md_text, word_count, page_count = convert_to_markdown(pdf_path)
                conv_duration = time.time() - t_conv

                if len(raw_md_text) < MIN_MD_CHARS:
                    status("    ⏭ SKIP: Converted text too short (%d chars)", len(raw_md_text))
                    stats["skipped"] += 1
                    pdf_path.unlink(missing_ok=True)
                    continue

                status("    ⚡ Converted: %d words, %d pages in %.1fs", word_count, page_count, conv_duration)

                # Step 4: Companion Code (Optional)
                if not paper.code_repo:
                    paper.code_repo = extract_code_repo(raw_md_text, paper.abstract)

                code_readme = None
                if paper.code_repo:
                    status("    📦 Companion code detected: %s", paper.code_repo)
                    code_readme = fetch_companion_code(paper.code_repo, gh_token)

                # Step 5: Strip Bibliography & Clean Text
                body_content, bibliography = strip_bibliography(raw_md_text)

                # Append companion code if present
                full_raw_text = body_content
                if code_readme:
                    full_raw_text += code_readme

                # Step 6: Apply ZERO-LINK Sanitization Engine
                status("    ✨ Sanitizing text: stripping all links, raw URLs, and badges...")
                clean_body = strip_all_links(full_raw_text)

                # Step 7: 100% Full-Coverage Smart Chunking for Fine-Tuning
                kb_id = next_kb_id(reg)
                paper.track = batch_name
                chunk_records = chunk_document_full_coverage(clean_body, kb_id, paper)
                total_chunks = len(chunk_records)
                status(f"    🧩 Full-Coverage Chunker: Created {total_chunks} chunks (100% retained, 0% truncated)")

                # Step 8: Build Zero-Link Markdown Document
                frontmatter = build_zero_link_frontmatter(paper, kb_id, word_count, page_count, total_chunks)
                final_clean_md = frontmatter + clean_body

                # Save local files
                md_path = TEXT_VAULT_DIR / f"{kb_id}.md"
                md_path.write_text(final_clean_md, encoding="utf-8")

                # Save dataset files
                paper_dataset_file = DATASET_VAULT_DIR / f"{kb_id}_dataset.jsonl"
                dataset_lines = [json.dumps(rec, ensure_ascii=False) for rec in chunk_records]
                paper_dataset_file.write_text("\n".join(dataset_lines) + "\n", encoding="utf-8")

                # Append to cumulative dataset
                with open(GLOBAL_DATASET_FILE, "a", encoding="utf-8") as gf:
                    gf.write("\n".join(dataset_lines) + "\n")

                # Rename PDF with KB ID for delivery
                kb_pdf_path = PDF_DIR / f"{kb_id}.pdf"
                if pdf_path.exists():
                    pdf_path.rename(kb_pdf_path)

                # Step 9: Multi-Topic Telegram Delivery
                status("    📤 Delivering to Telegram: PDF (#%d) | Text (#%d) | Dataset (#%d)...",
                       raw_topic, md_topic, dataset_topic)

                # Delivery A: Raw PDF -> Topic 354
                pdf_card = build_pdf_card(paper, kb_id)
                if kb_pdf_path.exists():
                    if pdf_size_mb < 50:
                        tg_send_doc(token, group_id, kb_pdf_path, pdf_card, message_thread_id=raw_topic)
                    else:
                        tg_send(token, group_id, pdf_card + f"\n\n📥 <i>PDF exceeds 50MB ({pdf_size_mb:.0f}MB)</i>",
                                message_thread_id=raw_topic)

                # Delivery B: Clean Text -> Topic 355
                text_card = build_text_card(paper, kb_id, word_count, page_count)
                tg_send_doc(token, group_id, md_path, text_card, message_thread_id=md_topic)

                # Delivery C: Dataset JSONL -> Topic 356
                dataset_card = build_dataset_card(paper, kb_id, total_chunks, word_count)
                tg_send_doc(token, group_id, paper_dataset_file, dataset_card, message_thread_id=dataset_topic)

                # Step 10: Register Paper Locally
                reg["papers"].append(paper.arxiv_id)
                reg["hashes"].append(pdf_hash)
                reg["titles"].append(paper.title)
                save_registry(reg)

                # Step 11: GitHub Dual-Vault Sync (text_vault/ + dataset_vault/ + README.md)
                try:
                    status("    🐙 Pushing %s to GitHub Dual-Vault...", kb_id)
                    github_save_vault(
                        gh_token=gh_token,
                        repo_url=gh_repo,
                        kb_id=kb_id,
                        clean_md_text=final_clean_md,
                        dataset_jsonl_text="\n".join(dataset_lines) + "\n",
                        reg=reg,
                    )
                    status("    🐙 GitHub Vault Sync OK")
                except Exception as gh_exc:
                    status("    ⚠️ GitHub push failed (non-fatal): %s", gh_exc)
                    stats.setdefault("gh_errors", 0)
                    stats["gh_errors"] += 1

                stats["added"] += 1
                status("    ✅ SUCCESS: %s archived and delivered!", kb_id)

            except Exception as paper_exc:
                stats["errors"] += 1
                error_msg = f"❌ [ERROR on {paper.arxiv_id}] {paper.title[:50]}: {paper_exc}"
                status("    %s", error_msg)

                # Send error alert to General Topic (#1)
                send_general_alert(
                    token=token,
                    chat_id=group_id,
                    text=f"⚠️ <b>Ingestion Error Detected</b>\n<b>Paper:</b> {paper.arxiv_id}\n<b>Title:</b> {paper.title[:60]}\n<b>Error:</b> <code>{html.escape(str(paper_exc)[:200])}</code>",
                    general_topic=general_topic,
                )

                # Rollback registry
                if paper.arxiv_id in reg.get("papers", []):
                    reg["papers"].remove(paper.arxiv_id)
                if pdf_hash and pdf_hash in reg.get("hashes", []):
                    reg["hashes"].remove(pdf_hash)
                if paper.title in reg.get("titles", []):
                    reg["titles"].remove(paper.title)
                save_registry(reg)

            finally:
                # Cleanup temporary PDF
                if pdf_path and pdf_path.exists():
                    pdf_path.unlink(missing_ok=True)
                kb_temp_pdf = PDF_DIR / f"{kb_id}.pdf"
                if kb_temp_pdf.exists():
                    kb_temp_pdf.unlink(missing_ok=True)

        time.sleep(1.0)

    total_time = round(time.time() - start_time)
    status("\n════════════════════════════════════════════════════════════════════════════")
    status("🎉 Cycle Complete in %ds: Scanned %d | Added %d | Skipped %d | Errors %d",
           total_time, stats["scanned"], stats["added"], stats["skipped"], stats["errors"])
    status("════════════════════════════════════════════════════════════════════════════")

    # Send Cycle Summary to General Topic (#1)
    summary_msg = (
        f"🤖 <b>Knowledge Agent v3 — Cycle Complete</b>\n"
        f"⏱ <b>Duration:</b> {total_time}s\n\n"
        f"📊 <b>Statistics:</b>\n"
        f"  • 🔍 Scanned: <b>{stats['scanned']}</b>\n"
        f"  • ✅ Ingested: <b>{stats['added']}</b>\n"
        f"  • ⏭ Skipped (Duplicates): <b>{stats['skipped']}</b>\n"
        f"  • ⚠️ Errors: <b>{stats['errors']}</b>\n\n"
        f"📁 <i>Text Vault, Dataset Vault, and Topics 354-356 up to date.</i>"
    )
    send_general_alert(token, group_id, summary_msg, general_topic)

    return stats


# ══════════════════════════════════════════════════════════════════════════════
#  CONFIGURATION
# ══════════════════════════════════════════════════════════════════════════════

DEFAULT_CONFIG = {
    "GITHUB_TOKEN": "YOUR_GITHUB_TOKEN",
    "GITHUB_REPO_URL": "https://github.com/Rawknowledge-database/knowledge",
    "TELEGRAM_BOT_TOKEN": "8525850416:AAGtYIM1sg8MF21_8lI2hOS1E-i9MosV4RE",
    "TELEGRAM_GROUP_ID": "-1003958148223",
    "RAW_PDF_TOPIC_ID": "354",
    "TEXT_MD_TOPIC_ID": "355",
    "DATASET_TOPIC_ID": "356",
    "GENERAL_TOPIC_ID": "1",
    "ADMIN_CHAT_ID": "6190001521",
    "HF_TOKEN": "YOUR_HF_TOKEN",
    "PAPERS_PER_CATEGORY": "15",
}


def prompt_env() -> dict:
    env_file = ROOT / ".env"
    file_cfg = {}
    if env_file.exists():
        for line in env_file.read_text(encoding="utf-8").splitlines():
            line = line.strip()
            if line and not line.startswith("#") and "=" in line:
                k, v = line.split("=", 1)
                file_cfg[k.strip()] = v.strip().strip("'\"")

    colab_secrets = {}
    try:
        from google.colab import userdata
        for k in DEFAULT_CONFIG.keys():
            try:
                secret_val = userdata.get(k)
                if secret_val:
                    colab_secrets[k] = str(secret_val).strip()
            except Exception:
                pass
    except Exception:
        pass

    env = {}
    for k, default_v in DEFAULT_CONFIG.items():
        val = (
            os.environ.get(k, "").strip()
            or colab_secrets.get(k, "")
            or str(file_cfg.get(k, "")).strip()
            or default_v
        )
        env[k] = val
    if env.get("HF_TOKEN"):
        os.environ["HF_TOKEN"] = env["HF_TOKEN"]
        os.environ["HUGGING_FACE_HUB_TOKEN"] = env["HF_TOKEN"]
    return env


# ══════════════════════════════════════════════════════════════════════════════
#  24/7 CRASH-PROOF DAEMON LOOP
# ══════════════════════════════════════════════════════════════════════════════

def daemon_loop(env: dict) -> None:
    """Run pipeline continuously 24/7 with zero sleeping — non-stop scan, convert, and upload."""
    cycle = 1

    status("════════════════════════════════════════════════════════════════════════════")
    status("⚡ Knowledge Agent v3 — 24/7 Autonomous Daemon Started")
    status("   • PDF Topic     : https://t.me/c/3958148223/%s", env.get("RAW_PDF_TOPIC_ID", "354"))
    status("   • Text Topic    : https://t.me/c/3958148223/%s", env.get("TEXT_MD_TOPIC_ID", "355"))
    status("   • Dataset Topic : https://t.me/c/3958148223/%s", env.get("DATASET_TOPIC_ID", "356"))
    status("   • General Topic : https://t.me/c/3958148223/%s", env.get("GENERAL_TOPIC_ID", "1"))
    status("════════════════════════════════════════════════════════════════════════════")

    # Broadcast startup to General Topic
    token = env["TELEGRAM_BOT_TOKEN"]
    group_id = env["TELEGRAM_GROUP_ID"]
    general_topic = int(env.get("GENERAL_TOPIC_ID", 1))
    send_general_alert(
        token, group_id,
        "🟢 <b>Knowledge Agent v3 Online (24/7 Autonomous Daemon)</b>\n"
        "• Tracking 10 research & cyber curriculum tracks\n"
        "• Multi-topic routing active (PDF: 354, Text: 355, Dataset: 356)\n"
        "• Zero-Link sanitizer & Full-Coverage chunking enabled.",
        general_topic,
    )

    while True:
        cycle_start = time.time()
        now_str = datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M:%S UTC")
        status("\n🚀 [Cycle #%d] Starting continuous scan at %s", cycle, now_str)

        try:
            stats = run(env)
            status("✅ [Cycle #%d] Completed in %ds: %s", cycle, round(time.time() - cycle_start), stats)
        except KeyboardInterrupt:
            status("\n🛑 Continuous engine stopped by user.")
            send_general_alert(token, group_id, "🛑 <b>Knowledge Agent paused by user.</b>", general_topic)
            break
        except Exception as cycle_exc:
            log.error("Cycle #%d encountered error: %s", cycle, cycle_exc, exc_info=True)
            status("⚠️ [Cycle #%d] Encountered error: %s. Recovering and continuing...", cycle, cycle_exc)
            send_general_alert(
                token, group_id,
                f"🚨 <b>Cycle #{cycle} Recovered from Error</b>\n<code>{html.escape(str(cycle_exc)[:250])}</code>\nRestarting scan immediately...",
                general_topic,
            )

        cycle += 1
        status("🔄 Starting Cycle #%d immediately...\n", cycle)
        time.sleep(3)


if __name__ == "__main__":
    collected_env = prompt_env()
    if "--once" in sys.argv or os.environ.get("RUN_ONCE", "").lower() in ("true", "1", "yes"):
        try:
            run(collected_env)
        except KeyboardInterrupt:
            print("\n🛑 Pipeline paused by user. Clean exit.")
    else:
        daemon_loop(collected_env)


In [ ]:
# ── Step 4: Run 24/7 Continuous Daemon ────────────────────────────
!python knowledge_agent.py
